# Movie Ticket Search with Playwright MCP Server

This notebook demonstrates using Neo's agentic workflow with the Playwright MCP server to automate movie ticket searches on AMC Theatres.

## ⚠️ IMPORTANT: Replace Cookies Before Running!

**You MUST replace the cookies in `amc_cookies.json` with fresh ones from your browser.**

The provided `amc_cookies.json` file contains placeholder values. To bypass Cloudflare bot protection:

### Step 1: Get Fresh Cookies
1. Visit https://www.amctheatres.com in Chrome/Firefox
2. Open DevTools (F12) → Application/Storage → Cookies
3. Export all cookies for `.amctheatres.com`

### Step 2: Using Browser Extension (Recommended)
1. Install "EditThisCookie" or "Cookie-Editor" extension
2. Visit AMC website and let it fully load
3. Click the extension icon → Export → Copy to clipboard
4. **Paste into `amc_cookies.json` in this folder**

### Step 3: Verify Critical Cookies
Make sure you have these cookies:
- `__cf_bm` - Cloudflare bot management (MOST IMPORTANT)
- `osano_consentmanager_uuid` - Consent tracking
- `osano_consentmanager` - Consent details
- `connect.sid` - Session ID
- `_ga`, `_gcl_au` - Analytics

**Note:** Cookies expire quickly (some in 30 minutes)! Get fresh ones before each session.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup MCP Client with Playwright

First, initialize the Playwright MCP client.

In [ ]:
import os
import json
from neo.mcp.client import MCPClient

# Load cookies from JSON file
with open("amc_cookies.json", "r") as f:
    raw_cookies = json.load(f)

# Convert browser extension cookie format to Playwright storage state format
storage_state = {
    "cookies": []
}

for cookie in raw_cookies:
    # Convert expirationDate (Unix timestamp) to expires (seconds from now, or -1 for session)
    expires = -1
    if "expirationDate" in cookie and cookie["expirationDate"]:
        expires = cookie["expirationDate"]
    
    # Convert sameSite format
    same_site = "Lax"  # default
    if "sameSite" in cookie:
        if cookie["sameSite"] == "no_restriction":
            same_site = "None"
        elif cookie["sameSite"] == "lax":
            same_site = "Lax"
        elif cookie["sameSite"] == "strict":
            same_site = "Strict"
        elif cookie["sameSite"] == "unspecified":
            same_site = "Lax"
    
    storage_state["cookies"].append({
        "name": cookie["name"],
        "value": cookie["value"],
        "domain": cookie["domain"],
        "path": cookie["path"],
        "expires": expires,
        "httpOnly": cookie.get("httpOnly", False),
        "secure": cookie.get("secure", False),
        "sameSite": same_site
    })

# Save storage state to file
storage_state_path = "amc_storage_state.json"
with open(storage_state_path, "w") as f:
    json.dump(storage_state, f, indent=2)

print(f"✅ Converted {len(storage_state['cookies'])} cookies to storage state format")

# Initialize Playwright MCP client with storage state
playwright_client = MCPClient(
    name="playwright",
    command="npx",
    args=[
        "-y", 
        "@playwright/mcp@latest",
        f"--storage-state={storage_state_path}"  # Pass storage state file
    ]
)

# Connect to the server
await playwright_client.aconnect()

# List available tools
print("\nAvailable Playwright tools:")
for tool_name in playwright_client.tools.keys():
    print(f"  - {tool_name}")

print(f"\n✅ Playwright MCP server started with storage state from {storage_state_path}")

## Use Neo Agentic Workflow

Create a Neo workflow with a task to search for movie tickets.

## Anti-Bot Protection Strategies

To bypass AMC's Cloudflare bot protection, we'll use techniques from the working AMC crawler:
1. **Cookies**: Pre-load session cookies (especially Cloudflare's `__cf_bm`)
2. **Random delays**: Wait 3-6 seconds between actions
3. **Mouse movements**: Simulate human-like cursor movement
4. **Wait strategies**: Properly wait for elements and page loads

First, let's set up cookies and configure the browser with human-like behavior.

### How Cookies Are Loaded

Cookies are automatically loaded by the Playwright MCP server via the `--storage-state` argument:

1. **Conversion**: Browser extension cookies (`amc_cookies.json`) are converted to Playwright storage state format
2. **Storage State File**: Saved as `amc_storage_state.json` with the format:
   ```json
   {
     "cookies": [{
       "name": "...",
       "value": "...",
       "domain": "...",
       "expires": ...,
       "httpOnly": true,
       "secure": true,
       "sameSite": "Lax"
     }]
   }
   ```
3. **MCP Client Init**: The storage state file is passed via command-line argument:
   ```python
   playwright_client = MCPClient(
       name="playwright",
       command="npx",
       args=["-y", "@playwright/mcp@latest", "--storage-state=amc_storage_state.json"]
   )
   ```

The browser will have all cookies automatically loaded before any navigation, so the agent doesn't need to manually load them.

In [ ]:
from neo.agentic.neo import Neo
from neo.agentic.task import ModelTask
from neo.agentic.instruction import Instruction, ModelConfigs, OtherConfigs

# Create a task for web browsing with anti-bot strategies
browsing_task = ModelTask(
    id="search_movie_tickets",
    user_input="""Go to the AMC website and search for Avatar movie showtimes near Rancho Cucamonga, CA. 

Important anti-bot strategies:
1. Cookies are already loaded via storage state - no need to load them manually
2. Add random delays (3-6 seconds) between actions using browser_wait_for
3. Use mouse movements: page.mouse.move(randomX, randomY) via browser_run_code
4. Always wait for elements to load before interacting
5. Check for redirects to about:blank (indicates bot detection)

List the available showtimes.""",
    instruction=Instruction(
        model_configs=ModelConfigs(
            model="gpt-5.2",
            
        ),
        content="""You are a sophisticated web browsing assistant specializing in bypassing bot protection.

Cookies are already loaded via the Playwright storage state - the browser already has all cookies set.

Key strategies:
- Use realistic delays between actions (3-6 seconds) with browser_wait_for
- Simulate human-like mouse movements with random coordinates using browser_run_code
- Wait for page elements using proper selectors
- Check for redirects to about:blank (bot detection indicator)
- Navigate directly to theater showtime pages""",
        other_configs=OtherConfigs(
            mcp_clients=[playwright_client]
        )
    ),
)

# Create Neo instance and run the task
neo = Neo(
    tasks=browsing_task,
    max_tool_execution_rounds=15  # Allow more rounds for anti-bot actions
)

# Execute the task
result_thread = await neo.run_all()

# Display results
result_thread.display()

In [ ]:
result_thread[-1]

## View Task Status

Check the status of all tasks.

In [ ]:
neo.display_task_status()

## Cleanup

Close the MCP client connection.

In [ ]:
# Close MCP client connection
await playwright_client.aclose()
print("Disconnected from Playwright MCP server")

## Notes

**How it works:**
- Neo manages task execution with the AI model
- The model automatically uses Playwright MCP tools to navigate websites
- Tasks can be chained together to build complex workflows
- The framework handles tool execution rounds and error handling

**Cookie Management:**
- Browser extension cookies (`amc_cookies.json`) are converted to Playwright storage state format
- Storage state is passed to the Playwright MCP server via `--storage-state` argument
- Cookies are automatically loaded in the browser before any navigation
- No manual cookie loading required in the agent's code

**Anti-Bot Protection:**
Based on the working AMC crawler (`src/discord-bot/discord_bot/amc_crawler/`), we use:
1. **Fresh Cookies** - Especially `__cf_bm` (Cloudflare), obtained from a real browser session
2. **Random Delays** - 3-6 seconds between actions using `browser_wait_for`
3. **Mouse Movements** - Simulate human-like cursor movement with `browser_run_code`
4. **Direct Theater URLs** - Navigate directly to theater pages instead of searching
5. **Proper Waits** - Use `wait_for_selector` and check for redirects to `about:blank`

**Important:**
- Cookies expire! Get fresh ones before each session
- Cookies are automatically loaded via storage state - agent doesn't need to load them manually
- Increase `max_tool_execution_rounds` to 15+ for complex anti-bot workflows
- Check if page redirects to `about:blank` (indicates bot detection)
- Use specific theater URLs: `https://www.amctheatres.com/movie-theatres/los-angeles/{theater}/showtimes`

**Troubleshooting:**
- If redirected to `about:blank` → Get fresh cookies
- If Cloudflare challenge appears → Add longer delays and more mouse movements
- If no showtimes found → Verify the theater slug and date in the URL
- If `process is not defined` error → Cookies should be loaded via `--storage-state`, not browser-side code